# Estudo técnico — tabelas DB2 de transações, contas, complementos e categorias

**Objetivo:** observar os dados reais antes de qualquer nova inferência financeira.

Este notebook é independente do painel e não altera nenhuma tabela. Ele somente:

`consulta → conta → agrupa → mostra valores → testa chaves → testa relacionamentos → mostra inconsistências`

## Escopo

Somente estas cinco tabelas:

1. `DB2GFP.TRAN_RLZD_INST_PCT`
2. `DB2GFP.INF_OPB_CT_CLI`
3. `DB2GFP.CMPT_TRAN_RLZD_CC`
4. `DB2GFP.CTGR_TRAN_OPB`
5. `DB2GFP.GR_CTGR_TRAN`

Recorte inicial:

- período: **01/07/2026 a 31/07/2026**
- moeda: **BRL**
- cliente: informado de forma protegida

## Regra de segurança do cliente

O valor do identificador do cliente:

- não é impresso;
- não é incluído em DataFrame exibido;
- não é interpolado em SQL;
- não é exportado;
- não é usado em nome de arquivo;
- não é transformado em hash para compartilhamento.

As consultas usam **parâmetros DB2 (`?`)**. As funções de exibição também removem a coluna sensível e mascaram qualquer ocorrência acidental do valor protegido em outra coluna.

## Princípio metodológico

**Nome de coluna não é evidência suficiente.**

O fluxo é deliberadamente em duas fases:

1. observar catálogo, restrições, índices, amostras e valores;
2. somente depois registrar hipóteses de grão, chaves e relacionamentos e testá-las.

Nenhuma regra de salário, gasto, principalidade ou categorização financeira é aplicada aqui.

## 1. Conexão DB2 local

Para este estudo exploratório, o recorte é pequeno e as agregações são executadas no próprio DB2. O notebook recebe apenas resultados controlados.

Não há criação de sessão Spark neste notebook.

In [ ]:
import re
from datetime import date
from getpass import getpass

import pandas as pd
from IPython.display import display
from bbmagic import Db2

db2 = Db2()

SCHEMA = "DB2GFP"
TABELAS = [
    "TRAN_RLZD_INST_PCT",
    "INF_OPB_CT_CLI",
    "CMPT_TRAN_RLZD_CC",
    "CTGR_TRAN_OPB",
    "GR_CTGR_TRAN",
]

DATA_INICIO = date(2026, 7, 1)
DATA_FIM_EXCLUSIVO = date(2026, 8, 1)
MOEDA_ESTUDO = "BRL"

print("DB2 configurado.")
print(f"Período: {DATA_INICIO:%d/%m/%Y} a 31/07/2026")
print(f"Moeda: {MOEDA_ESTUDO}")

## 2. Entrada protegida do cliente

A entrada usa `getpass`, portanto o valor digitado não aparece na célula nem no output.

**Não substitua esta célula por uma atribuição literal com o identificador.**

In [ ]:
_cliente_protegido = getpass("Informe o cliente protegido: ").strip()

if not _cliente_protegido:
    raise ValueError("Identificador protegido não informado.")

print("Cliente protegido carregado em memória.")

## 3. Funções de segurança e execução

In [ ]:
_ID_RE = re.compile(r"^[A-Z][A-Z0-9_]*$")

def _id(nome: str) -> str:
    # Valida identificadores de schema/tabela/coluna antes de inseri-los no SQL.
    nome = str(nome).strip().upper()
    if not _ID_RE.fullmatch(nome):
        raise ValueError("Identificador DB2 inválido.")
    return nome

def _tabela(nome: str) -> str:
    nome = _id(nome)
    if nome not in TABELAS:
        raise ValueError("Tabela fora do escopo deste notebook.")
    return f"{SCHEMA}.{nome}"

def _proteger_df(df: pd.DataFrame) -> pd.DataFrame:
    # Remove coluna sensível e mascara ocorrência acidental do valor protegido.
    if not isinstance(df, pd.DataFrame):
        return df

    out = df.copy()

    cols_drop = [c for c in out.columns if str(c).strip().upper() == "CD_CLI"]
    if cols_drop:
        out = out.drop(columns=cols_drop)

    segredo = str(_cliente_protegido)
    if segredo:
        for c in out.columns:
            out[c] = out[c].map(
                lambda x: "<PROTEGIDO>"
                if pd.notna(x) and segredo in str(x)
                else x
            )
    return out

def mostrar(df: pd.DataFrame, n=None):
    seguro = _proteger_df(df)
    if n is not None:
        seguro = seguro.head(int(n))
    display(seguro)

def consultar(sql: str, params=None) -> pd.DataFrame:
    # SQL e parâmetros nunca são impressos.
    try:
        return db2.query(sql, params=params or [], chunksize=None)
    except Exception:
        raise RuntimeError(
            "Falha na consulta DB2. Revise acesso, metadados ou mapeamento. "
            "SQL e parâmetros não são exibidos por segurança."
        ) from None

## 4. Metadados reais das cinco tabelas

Primeiro observamos o catálogo. Esta etapa não consulta registros de cliente.

O relatório mostra, conforme disponível no `SYSCAT.SYSCOLUMNS`, nome físico, tipo, nulabilidade e comentário/descrição das colunas.

In [ ]:
META = {}

for tabela in TABELAS:
    meta = db2.syscolumns(SCHEMA, tabela)
    META[tabela] = meta.copy()

    print("\n" + "=" * 100)
    print(f"{SCHEMA}.{tabela}")
    print("=" * 100)

    preferidas = [
        "COLNO", "COLNAME", "TYPENAME", "LENGTH", "SCALE",
        "NULLS", "DEFAULT", "REMARKS"
    ]
    cols = [c for c in preferidas if c in meta.columns]
    mostrar(meta[cols] if cols else meta)

## 5. Estatísticas de catálogo

Essas estatísticas ajudam a entender volumetria e características físicas sem varrer as tabelas.

In [ ]:
for tabela in TABELAS:
    print("\n" + "=" * 100)
    print(f"Estatísticas: {SCHEMA}.{tabela}")
    print("=" * 100)
    stats = db2.systabstats(SCHEMA, tabela)
    mostrar(stats)

## 6. Chaves, restrições e índices declarados

Antes de inferir qualquer chave pelos valores, verificamos se o catálogo DB2 declara PK/UK/FK e índices.

Ausência de restrição declarada **não prova** ausência de chave lógica; apenas significa que ela não está formalizada nessa camada do catálogo.

In [ ]:
def catalogo_restricoes(tabela: str) -> pd.DataFrame:
    sql = '''
        SELECT
            TC.TYPE AS TIPO_RESTRICAO,
            TC.CONSTNAME AS NOME_RESTRICAO,
            K.COLSEQ AS ORDEM_COLUNA,
            K.COLNAME AS COLUNA
        FROM SYSCAT.TABCONST TC
        LEFT JOIN SYSCAT.KEYCOLUSE K
          ON K.TABSCHEMA = TC.TABSCHEMA
         AND K.TABNAME = TC.TABNAME
         AND K.CONSTNAME = TC.CONSTNAME
        WHERE TC.TABSCHEMA = ?
          AND TC.TABNAME = ?
        ORDER BY TC.TYPE, TC.CONSTNAME, K.COLSEQ
    '''
    return consultar(sql, [SCHEMA, _id(tabela)])

def catalogo_referencias(tabela: str) -> pd.DataFrame:
    sql = '''
        SELECT
            CONSTNAME,
            REFTABSCHEMA,
            REFTABNAME,
            REFKEYNAME
        FROM SYSCAT.REFERENCES
        WHERE TABSCHEMA = ?
          AND TABNAME = ?
        ORDER BY CONSTNAME
    '''
    return consultar(sql, [SCHEMA, _id(tabela)])

def catalogo_indices(tabela: str) -> pd.DataFrame:
    sql = '''
        SELECT
            INDSCHEMA,
            INDNAME,
            UNIQUERULE,
            COLNAMES
        FROM SYSCAT.INDEXES
        WHERE TABSCHEMA = ?
          AND TABNAME = ?
        ORDER BY UNIQUERULE DESC, INDNAME
    '''
    return consultar(sql, [SCHEMA, _id(tabela)])

for tabela in TABELAS:
    print("\n" + "=" * 100)
    print(f"Catálogo estrutural: {SCHEMA}.{tabela}")
    print("=" * 100)

    print("\nRestrições:")
    mostrar(catalogo_restricoes(tabela))

    print("\nReferências declaradas:")
    mostrar(catalogo_referencias(tabela))

    print("\nÍndices:")
    mostrar(catalogo_indices(tabela))

# Fase 2 — mapeamento somente após observar evidências

A célula abaixo é o **único ponto manual obrigatório**.

Preencha somente o que foi confirmado pelo catálogo e/ou por inspeção dos valores.

- Use `None` quando ainda não estiver confirmado.
- Use lista para papéis que podem ter mais de uma coluna.
- Não escolha uma coluna somente porque o nome “parece correto”.

In [ ]:
MAPEAMENTO = {
    "TRAN_RLZD_INST_PCT": {
        "cliente": "CD_CLI",
        "data": None,
        "moeda": None,
        "valor": None,
        "id_transacao": [],
        "conta": [],
        "instituicao": [],
        "categoria": [],
        "contraparte": [],
        "texto_categoria": [],
        "texto_contraparte": [],
    },

    "INF_OPB_CT_CLI": {
        "cliente": None,
        "conta": [],
        "instituicao": [],
        "outras_chaves": [],
    },

    "CMPT_TRAN_RLZD_CC": {
        "id_transacao": [],
        "conta": [],
        "instituicao": [],
        "contraparte": [],
        "texto_contraparte": [],
    },

    "CTGR_TRAN_OPB": {
        "categoria": [],
        "grupo": [],
        "descricao_categoria": [],
        "outras_chaves": [],
    },

    "GR_CTGR_TRAN": {
        "grupo": [],
        "descricao_grupo": [],
        "outras_chaves": [],
    },
}

print(
    "MAPEAMENTO carregado. "
    "Preencha somente colunas já confirmadas pela evidência observada acima."
)

## 7. Validação do mapeamento contra o catálogo

In [ ]:
def colunas_catalogo(tabela: str) -> set[str]:
    meta = META[_id(tabela)]
    coluna_nome = "COLNAME" if "COLNAME" in meta.columns else None
    if not coluna_nome:
        raise ValueError(f"Não foi possível localizar COLNAME no catálogo de {tabela}.")
    return set(meta[coluna_nome].astype(str).str.upper())

def validar_coluna(tabela: str, coluna):
    if coluna is None:
        return
    col = _id(coluna)
    if col not in colunas_catalogo(tabela):
        raise ValueError(f"Coluna configurada não existe em {tabela}.")

def validar_mapeamento():
    for tabela, papeis in MAPEAMENTO.items():
        for _, valor in papeis.items():
            if isinstance(valor, list):
                for col in valor:
                    validar_coluna(tabela, col)
            else:
                validar_coluna(tabela, valor)
    print("Mapeamento: todas as colunas preenchidas existem no catálogo.")

validar_mapeamento()

## 8. Construção segura do recorte de julho/BRL

Esta etapa só roda quando `data` e `moeda` da tabela central tiverem sido confirmadas.

O identificador do cliente é enviado ao DB2 como parâmetro. Ele não é concatenado ao SQL.

In [ ]:
def filtro_transacoes(alias="T"):
    cfg = MAPEAMENTO["TRAN_RLZD_INST_PCT"]

    if not cfg["data"] or not cfg["moeda"]:
        raise ValueError(
            "Antes de consultar julho/BRL, confirme no MAPEAMENTO "
            "as colunas 'data' e 'moeda' da tabela central."
        )

    a = _id(alias)
    c_cli = _id(cfg["cliente"])
    c_data = _id(cfg["data"])
    c_moeda = _id(cfg["moeda"])

    where = (
        f"{a}.{c_cli} = ? "
        f"AND {a}.{c_data} >= ? "
        f"AND {a}.{c_data} < ? "
        f"AND {a}.{c_moeda} = ?"
    )
    params = [
        _cliente_protegido,
        DATA_INICIO,
        DATA_FIM_EXCLUSIVO,
        MOEDA_ESTUDO,
    ]
    return where, params

where_trans, params_trans = filtro_transacoes()
print("Filtro de estudo validado: cliente protegido + julho/2026 + BRL.")

## 9. Quantidade de registros no recorte

In [ ]:
sql = f'''
    SELECT COUNT(*) AS QT_REGISTROS
    FROM {_tabela("TRAN_RLZD_INST_PCT")} T
    WHERE {where_trans}
'''
mostrar(consultar(sql, params_trans))

## 10. Amostra controlada da tabela central

A amostra exclui explicitamente `CD_CLI` da projeção. Qualquer ocorrência acidental do valor protegido em outra coluna também é mascarada pela função `mostrar`.

In [ ]:
def colunas_seguras_para_amostra(tabela: str) -> list[str]:
    cols = sorted(colunas_catalogo(tabela))
    return [c for c in cols if c != "CD_CLI"]

cols = colunas_seguras_para_amostra("TRAN_RLZD_INST_PCT")
projecao = ", ".join(f"T.{_id(c)}" for c in cols)

sql = f'''
    SELECT {projecao}
    FROM {_tabela("TRAN_RLZD_INST_PCT")} T
    WHERE {where_trans}
    FETCH FIRST 50 ROWS ONLY
'''
mostrar(consultar(sql, params_trans))

# Fase 3 — perfil de valores

O objetivo aqui é observar **o que realmente aparece**, sem interpretar financeiramente.

Use apenas colunas que já foram confirmadas no `MAPEAMENTO`.

In [ ]:
def frequencias_no_recorte(coluna: str, limite=100) -> pd.DataFrame:
    c = _id(coluna)
    limite = int(limite)
    if not (1 <= limite <= 500):
        raise ValueError("Limite deve ficar entre 1 e 500.")

    sql = f'''
        SELECT
            T.{c} AS VALOR,
            COUNT(*) AS QT_REGISTROS
        FROM {_tabela("TRAN_RLZD_INST_PCT")} T
        WHERE {where_trans}
        GROUP BY T.{c}
        ORDER BY QT_REGISTROS DESC
        FETCH FIRST {limite} ROWS ONLY
    '''
    return consultar(sql, params_trans)

cfg_t = MAPEAMENTO["TRAN_RLZD_INST_PCT"]
colunas_perfil = []

for papel in [
    "valor", "id_transacao", "conta", "instituicao",
    "categoria", "contraparte", "texto_categoria", "texto_contraparte"
]:
    v = cfg_t[papel]
    if isinstance(v, list):
        colunas_perfil.extend(v)
    elif v:
        colunas_perfil.append(v)

for coluna in dict.fromkeys(colunas_perfil):
    print("\n" + "=" * 100)
    print(f"Frequências observadas — {coluna}")
    print("=" * 100)
    mostrar(frequencias_no_recorte(coluna))

## 11. Nulos e cardinalidade das colunas confirmadas

Esses números ajudam a diferenciar:

- coluna identificadora;
- atributo de baixa cardinalidade;
- coluna opcional;
- coluna potencialmente inadequada como chave.

In [ ]:
def perfil_coluna_recorte(coluna: str) -> pd.DataFrame:
    c = _id(coluna)
    sql = f'''
        SELECT
            COUNT(*) AS QT_TOTAL,
            SUM(CASE WHEN T.{c} IS NULL THEN 1 ELSE 0 END) AS QT_NULOS,
            COUNT(DISTINCT T.{c}) AS QT_DISTINTOS
        FROM {_tabela("TRAN_RLZD_INST_PCT")} T
        WHERE {where_trans}
    '''
    return consultar(sql, params_trans)

for coluna in dict.fromkeys(colunas_perfil):
    print(f"Perfil: {coluna}")
    mostrar(perfil_coluna_recorte(coluna))

# Fase 4 — teste de grão

O grão não será “adivinhado”. Defina abaixo **hipóteses de chave** com base no que foi observado.

Exemplo de formato:

```python
HIPOTESES_GRAO = [
    ("TRAN_RLZD_INST_PCT", ["COL_A"]),
    ("TRAN_RLZD_INST_PCT", ["COL_A", "COL_B"]),
]
```

O teste mostra quantos grupos repetem e quantas linhas excedentes existem.

In [ ]:
HIPOTESES_GRAO = [
    # Preencher após observar os dados.
]

In [ ]:
def testar_grao_transacoes(colunas: list[str]) -> pd.DataFrame:
    if not colunas:
        raise ValueError("Informe pelo menos uma coluna.")

    cols = [_id(c) for c in colunas]
    for c in cols:
        validar_coluna("TRAN_RLZD_INST_PCT", c)

    group = ", ".join(f"T.{c}" for c in cols)

    sql = f'''
        SELECT
            COUNT(*) AS QT_GRUPOS_REPETIDOS,
            COALESCE(SUM(QT - 1), 0) AS QT_LINHAS_EXCEDENTES,
            COALESCE(MAX(QT), 0) AS MAIOR_REPETICAO
        FROM (
            SELECT {group}, COUNT(*) AS QT
            FROM {_tabela("TRAN_RLZD_INST_PCT")} T
            WHERE {where_trans}
            GROUP BY {group}
            HAVING COUNT(*) > 1
        ) X
    '''
    return consultar(sql, params_trans)

for tabela, colunas in HIPOTESES_GRAO:
    if _id(tabela) != "TRAN_RLZD_INST_PCT":
        print(f"Teste de grão direto ainda não configurado para {tabela}.")
        continue

    print(f"Hipótese de grão: {tabela} -> {colunas}")
    mostrar(testar_grao_transacoes(colunas))

## 12. Inspeção das duplicidades

Para uma hipótese de chave que repete, esta função mostra **somente as chaves da hipótese e a contagem**, nunca o cliente.

In [ ]:
def maiores_duplicidades_transacoes(colunas: list[str], limite=100) -> pd.DataFrame:
    cols = [_id(c) for c in colunas]
    group = ", ".join(f"T.{c}" for c in cols)
    proj = ", ".join(f"T.{c}" for c in cols)
    limite = max(1, min(int(limite), 500))

    sql = f'''
        SELECT {proj}, COUNT(*) AS QT
        FROM {_tabela("TRAN_RLZD_INST_PCT")} T
        WHERE {where_trans}
        GROUP BY {group}
        HAVING COUNT(*) > 1
        ORDER BY QT DESC
        FETCH FIRST {limite} ROWS ONLY
    '''
    return consultar(sql, params_trans)

# Exemplo de uso somente após escolher uma hipótese:
# mostrar(maiores_duplicidades_transacoes(["COL_A", "COL_B"]))

# Fase 5 — teste de relacionamentos

Também não assumimos joins.

Registre apenas relacionamentos candidatos que tenham sido sustentados por:

- catálogo/descrição;
- valores compatíveis;
- cardinalidade observada;
- contexto documental.

Formato:

```python
RELACIONAMENTOS = [
    {
        "nome": "transacao -> complemento",
        "esquerda": "TRAN_RLZD_INST_PCT",
        "cols_esquerda": ["COL_A"],
        "direita": "CMPT_TRAN_RLZD_CC",
        "cols_direita": ["COL_B"],
    }
]
```

O teste parte **somente das transações do cliente em julho/BRL** e mede:

- linhas/chaves da base;
- chaves sem correspondência;
- chaves com uma correspondência;
- chaves com mais de uma correspondência;
- maior multiplicidade encontrada.

In [ ]:
RELACIONAMENTOS = [
    # Preencher após confirmar candidatos.
]

In [ ]:
def testar_relacionamento(rel: dict) -> pd.DataFrame:
    esquerda = _id(rel["esquerda"])
    direita = _id(rel["direita"])

    if esquerda != "TRAN_RLZD_INST_PCT":
        raise ValueError(
            "Nesta primeira versão, o lado esquerdo deve ser a tabela central "
            "para preservar o recorte protegido de julho/BRL."
        )

    ce = [_id(c) for c in rel["cols_esquerda"]]
    cd = [_id(c) for c in rel["cols_direita"]]

    if not ce or len(ce) != len(cd):
        raise ValueError("As listas de colunas do join devem ter o mesmo tamanho e não ser vazias.")

    for c in ce:
        validar_coluna(esquerda, c)
    for c in cd:
        validar_coluna(direita, c)

    chave_base = ", ".join(f"T.{c} AS K{i}" for i, c in enumerate(ce, start=1))
    join_cte = " AND ".join(f"B.K{i} = D.{r}" for i, r in enumerate(cd, start=1))

    sql = f'''
        WITH BASE AS (
            SELECT {chave_base}
            FROM {_tabela(esquerda)} T
            WHERE {where_trans}
        ),
        MULT AS (
            SELECT
                {", ".join(f"B.K{i}" for i in range(1, len(ce) + 1))},
                COUNT(D.{cd[0]}) AS QT_MATCH
            FROM BASE B
            LEFT JOIN {_tabela(direita)} D
              ON {join_cte}
            GROUP BY {", ".join(f"B.K{i}" for i in range(1, len(ce) + 1))}
        )
        SELECT
            COUNT(*) AS QT_CHAVES_BASE,
            SUM(CASE WHEN QT_MATCH = 0 THEN 1 ELSE 0 END) AS QT_SEM_MATCH,
            SUM(CASE WHEN QT_MATCH = 1 THEN 1 ELSE 0 END) AS QT_MATCH_UNICO,
            SUM(CASE WHEN QT_MATCH > 1 THEN 1 ELSE 0 END) AS QT_MATCH_MULTIPLO,
            MAX(QT_MATCH) AS MAIOR_MULTIPLICIDADE
        FROM MULT
    '''
    return consultar(sql, params_trans)

for rel in RELACIONAMENTOS:
    print("\n" + "=" * 100)
    print(rel.get("nome", "Relacionamento"))
    print("=" * 100)
    mostrar(testar_relacionamento(rel))

## 13. Valores da dimensão relacionada para as transações do recorte

Depois que um relacionamento estiver confirmado, podemos observar valores da tabela relacionada sem trazer o cliente ao resultado.

In [ ]:
def valores_relacionados(rel: dict, coluna_direita: str, limite=100) -> pd.DataFrame:
    esquerda = _id(rel["esquerda"])
    direita = _id(rel["direita"])
    if esquerda != "TRAN_RLZD_INST_PCT":
        raise ValueError("O lado esquerdo deve ser a tabela central.")

    ce = [_id(c) for c in rel["cols_esquerda"]]
    cd = [_id(c) for c in rel["cols_direita"]]
    alvo = _id(coluna_direita)
    validar_coluna(direita, alvo)

    chave_base = ", ".join(f"T.{c} AS K{i}" for i, c in enumerate(ce, start=1))
    join_cte = " AND ".join(f"B.K{i} = D.{r}" for i, r in enumerate(cd, start=1))
    limite = max(1, min(int(limite), 500))

    sql = f'''
        WITH BASE AS (
            SELECT {chave_base}
            FROM {_tabela(esquerda)} T
            WHERE {where_trans}
        )
        SELECT
            D.{alvo} AS VALOR,
            COUNT(*) AS QT
        FROM BASE B
        JOIN {_tabela(direita)} D
          ON {join_cte}
        GROUP BY D.{alvo}
        ORDER BY QT DESC
        FETCH FIRST {limite} ROWS ONLY
    '''
    return consultar(sql, params_trans)

# Exemplo:
# mostrar(valores_relacionados(RELACIONAMENTOS[0], "COL_DESCRICAO"))

# Fase 6 — investigação específica de “Água”

Esta seção **não categoriza** a transação.

Ela apenas procura o texto “Água” em colunas textuais que tenham sido previamente confirmadas como descrição/rótulo de categoria ou contraparte.

Cadastre abaixo as colunas textuais confirmadas e a origem de cada uma.

In [ ]:
ALVOS_AGUA = [
    # Exemplo de estrutura após evidência:
    # {
    #     "relacionamento": RELACIONAMENTOS[0],
    #     "coluna_direita": "COL_DESCRICAO"
    # }
]

In [ ]:
def procurar_texto_relacionado(rel: dict, coluna_direita: str, termo="Água", limite=100) -> pd.DataFrame:
    esquerda = _id(rel["esquerda"])
    direita = _id(rel["direita"])
    ce = [_id(c) for c in rel["cols_esquerda"]]
    cd = [_id(c) for c in rel["cols_direita"]]
    alvo = _id(coluna_direita)
    validar_coluna(direita, alvo)

    chave_base = ", ".join(f"T.{c} AS K{i}" for i, c in enumerate(ce, start=1))
    join_cte = " AND ".join(f"B.K{i} = D.{r}" for i, r in enumerate(cd, start=1))
    limite = max(1, min(int(limite), 500))

    sql = f'''
        WITH BASE AS (
            SELECT {chave_base}
            FROM {_tabela(esquerda)} T
            WHERE {where_trans}
        )
        SELECT
            D.{alvo} AS VALOR_ENCONTRADO,
            COUNT(*) AS QT
        FROM BASE B
        JOIN {_tabela(direita)} D
          ON {join_cte}
        WHERE UPPER(CAST(D.{alvo} AS VARCHAR(1000))) LIKE UPPER(?)
        GROUP BY D.{alvo}
        ORDER BY QT DESC
        FETCH FIRST {limite} ROWS ONLY
    '''
    return consultar(sql, params_trans + [f"%{termo}%"])

for alvo in ALVOS_AGUA:
    print("\nBusca textual observacional: Água")
    mostrar(
        procurar_texto_relacionado(
            alvo["relacionamento"],
            alvo["coluna_direita"],
            termo="Água",
        )
    )

# Fase 7 — quadro de evidências para a próxima refatoração

Preencha este quadro **somente com fatos observados**.

Não registrar aqui conclusões financeiras.

In [ ]:
quadro_evidencias = pd.DataFrame(
    [
        {
            "TABELA": f"{SCHEMA}.{t}",
            "GRAO_OBSERVADO": "",
            "CHAVE_TRANSACAO": "",
            "CHAVE_CONTA": "",
            "CHAVE_INSTITUICAO": "",
            "CHAVE_CATEGORIA": "",
            "CHAVE_CONTRAPARTE": "",
            "DUPLICIDADES_OBSERVADAS": "",
            "NULOS_RELEVANTES": "",
            "RELACIONAMENTOS_CONFIRMADOS": "",
            "OBSERVACOES": "",
        }
        for t in TABELAS
    ]
)

mostrar(quadro_evidencias)

## Checklist de encerramento do estudo

Antes de retomar qualquer refatoração funcional, responder com evidência:

- [ ] Qual é o grão real de `TRAN_RLZD_INST_PCT` no recorte?
- [ ] Existe identificador único de transação? É realmente único?
- [ ] Qual coluna ou combinação identifica conta?
- [ ] Qual coluna ou combinação identifica instituição?
- [ ] Como `INF_OPB_CT_CLI` se relaciona com a tabela central? Qual cardinalidade?
- [ ] Como `CMPT_TRAN_RLZD_CC` se relaciona com a tabela central? Qual cardinalidade?
- [ ] Onde a contraparte aparece e com quais valores?
- [ ] Como categoria e grupo se relacionam às transações?
- [ ] Há chaves de dimensão duplicadas?
- [ ] Há transações sem correspondência nas tabelas auxiliares?
- [ ] O rótulo “Água” aparece em qual tabela/coluna e associado a quais registros?
- [ ] As duplicidades são duplicidade de transação ou multiplicidade provocada por join?
- [ ] Conta e instituição são atributos da transação, da conta, ou ambos?
- [ ] Todas as conclusões acima estão sustentadas por contagem/amostra/cardinalidade observada?

**Não exportar o DataFrame nem salvar dados do cliente fora do notebook.**